# 05 — Catalog Enrichment

Adds the catalog metadata that downstream tools (AI/BI, Genie, Discover) rely
on to surface the right thing automatically:

- **Comments** on every gold table and key column
- **Tags** — `domain=hockey_analytics`, `tier=bronze|silver|gold`, `data_owner`
- **Column-level tags** — `pii` flag on player birth fields (when ingested)
- A lineage check via `system.access.column_lineage` so you can see the bronze→silver→gold path

In [1]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [2]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

Catalog: alexander_booth
Bronze:  alexander_booth.sportlogiq_nhl_bronze
Silver:  alexander_booth.sportlogiq_nhl_silver
Gold:    alexander_booth.sportlogiq_nhl_gold
Volume:  /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data


In [3]:
B = f"{UC_CATALOG}.{BRONZE_SCHEMA}"
S = f"{UC_CATALOG}.{SILVER_SCHEMA}"
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"

DATA_OWNER = os.getenv("DATA_OWNER", "hockey_analytics_team")
DOMAIN     = os.getenv("DOMAIN",     "hockey_analytics")
print(f"Domain: {DOMAIN}; data_owner: {DATA_OWNER}")

Domain: hockey_analytics; data_owner: hockey_analytics_team


In [4]:
def safe_set_tags(target: str, tags: dict):
    """Apply tags but tolerate governed-tag policy errors."""
    pairs = ", ".join([f"'{k}' = '{v}'" for k, v in tags.items()])
    try:
        spark.sql(f"ALTER {target} SET TAGS ({pairs})")
        print(f"  tagged {target}: {tags}")
    except Exception as e:
        print(f"  could not tag {target}: {str(e)[:120]}")


# Catalog + schema-level tags for chargeback / discovery
for schema, tier in [(BRONZE_SCHEMA, "bronze"), (SILVER_SCHEMA, "silver"), (GOLD_SCHEMA, "gold")]:
    safe_set_tags(f"SCHEMA {UC_CATALOG}.{schema}",
                  {"domain": DOMAIN, "tier": tier, "data_owner": DATA_OWNER})

  could not tag SCHEMA alexander_booth.sportlogiq_nhl_bronze: [RequestId=88756edb-95ac-40a4-9062-e5b947622ac3 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag SCHEMA alexander_booth.sportlogiq_nhl_silver: [RequestId=dffc22b2-d51b-4e80-a4e2-d42c8ca72b47 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag SCHEMA alexander_booth.sportlogiq_nhl_gold: [RequestId=bc1aeef4-496e-47cd-80d7-3f8660b0a64e ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


In [5]:
# Gold table comments — short, business-readable. AI/BI + Genie use these directly.
TABLE_COMMENTS = {
    "dim_team":            "NHL teams as captured by SportLogiq. Type-1 dimension.",
    "dim_player":          "Skaters and goalies as captured by SportLogiq. Type-1 dimension.",
    "dim_venue":           "NHL arenas / playing venues. Type-1 dimension.",
    "dim_game":            "One row per scheduled NHL game with venue and team foreign keys.",
    "dim_date":            "Calendar date dimension covering 2020-01-01 through today + 90d.",
    "fact_game_events":    "Compiled on-ice events (shots, hits, faceoffs, possessions). x_coord/y_coord are SportLogiq rink coordinates.",
    "fact_player_shifts":  "Per-shift events (on-ice, off-ice, line changes).",
    "fact_player_game_metrics":  "Long-form per-game metric grid: one row per (game, player, topic, metric_key).",
    "fact_player_season_metrics":"Long-form season-to-date metric grid for player/team/goalie/opposingteam scopes.",
}
for t, desc in TABLE_COMMENTS.items():
    spark.sql(f"COMMENT ON TABLE {G}.{t} IS '{desc}'")
    print(f"  commented {G}.{t}")

  commented alexander_booth.sportlogiq_nhl_gold.dim_team


  commented alexander_booth.sportlogiq_nhl_gold.dim_player


  commented alexander_booth.sportlogiq_nhl_gold.dim_venue


  commented alexander_booth.sportlogiq_nhl_gold.dim_game


  commented alexander_booth.sportlogiq_nhl_gold.dim_date


  commented alexander_booth.sportlogiq_nhl_gold.fact_game_events


  commented alexander_booth.sportlogiq_nhl_gold.fact_player_shifts


  commented alexander_booth.sportlogiq_nhl_gold.fact_player_game_metrics


  commented alexander_booth.sportlogiq_nhl_gold.fact_player_season_metrics


In [6]:
# Column comments where Genie struggles without them — sport-specific terminology.
# Embedded single quotes are doubled (SQL string escape) so the COMMENT ON
# statement parses cleanly.
COLUMN_COMMENTS = [
    (f"{G}.fact_game_events.x_coord",
     "Rink x-coordinate from SportLogiq (centre-ice = 0). Used for shot maps."),
    (f"{G}.fact_game_events.y_coord",
     "Rink y-coordinate from SportLogiq. Used for shot maps."),
    (f"{G}.fact_game_events.zone",
     "Zone classification: O (offensive), D (defensive), N (neutral)."),
    (f"{G}.fact_game_events.is_shot",
     "TRUE if the event is any kind of shot (on-net, blocked, missed)."),
    (f"{G}.fact_game_events.is_goal",
     "TRUE for events whose outcome is goal."),
    (f"{G}.fact_player_game_metrics.topic_id",
     "SportLogiq metric_topic id. Join to silver_metric_topics for human-readable names."),
    (f"{G}.fact_player_game_metrics.metric_key",
     "Metric short key — e.g. corsi_for, fenwick_for, expected_goals. Look up display name in metric_topics."),
]
for col, txt in COLUMN_COMMENTS:
    safe_txt = txt.replace("'", "''")
    try:
        spark.sql(f"COMMENT ON COLUMN {col} IS '{safe_txt}'")
    except Exception as e:
        print(f"  skipped {col}: {str(e)[:80]}")
print("column comments applied.")

column comments applied.


In [7]:
# Tier + domain tags on the gold tables
for t in TABLE_COMMENTS:
    safe_set_tags(f"TABLE {G}.{t}",
                  {"domain": DOMAIN, "tier": "gold", "data_owner": DATA_OWNER})

  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.dim_team: [RequestId=16494b7f-c43f-48a4-8a35-61487ee7c1ea ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.dim_player: [RequestId=a9cc006c-93eb-439e-94c9-907b0b686f28 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.dim_venue: [RequestId=69cf33cf-df6b-44ce-b5eb-f0896b4f79da ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.dim_game: [RequestId=757da403-9e6d-439f-9492-e0c6b809cdcf ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.dim_date: [RequestId=f3f7ea8b-5639-487f-978b-158f4f8a44f8 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.fact_game_events: [RequestId=ce6463c9-c1e7-4801-a41c-43164f504b98 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.fact_player_shifts: [RequestId=606ed7e5-2848-4f35-ac0e-ad8921ae0760 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.fact_player_game_metrics: [RequestId=08fe73b0-bec1-4196-987f-28512c26fd3e ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


  could not tag TABLE alexander_booth.sportlogiq_nhl_gold.fact_player_season_metrics: [RequestId=9074b1be-8e56-42e7-94fd-a9e7774f5f53 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Tag value ho


## Lineage check

Confirms the column-level lineage graph SportLogiq → bronze → silver → gold is
fully connected. Run after at least one full pipeline pass; lineage entries
require the gold tables to have been queried at least once.

In [8]:
try:
    spark.sql(f"""
        SELECT source_table_full_name, target_table_full_name, source_column_name, target_column_name
        FROM system.access.column_lineage
        WHERE target_table_catalog = '{UC_CATALOG}'
          AND target_table_schema  = '{GOLD_SCHEMA}'
        ORDER BY target_table_full_name
        LIMIT 20
    """).show(truncate=False)
except Exception as e:
    print(f"system.access.column_lineage not available in this workspace: {e}")

+----------------------+----------------------+------------------+------------------+
|source_table_full_name|target_table_full_name|source_column_name|target_column_name|
+----------------------+----------------------+------------------+------------------+
+----------------------+----------------------+------------------+------------------+

